In [8]:
import numpy as np

from feature_optimization import FeatureOptimizer
import feature_opt_functions as funcs
from indices import *

import tracemalloc
import time

In [9]:
data = [None]*2
data[0] = np.load("dataset_cropped_64_npy\\TRAIN_HEALTHY_even.npy") # healthy
data[1] = np.load("dataset_cropped_64_npy\\TRAIN_STRESSED_even.npy") # stressed

In [10]:
def display_features(selected, encoder, id_mapping=None):
    mapping = {
        0: "B1",
        1: "B2",
        2: "B3",
        3: "B4",
        4: "B5",
        5: "B6",
        6: "B7",
        7: "B8",
        8: "B8A",
        9: "B9",
        10: "B11",
        11: "B12"
    }

    for id in selected:
        real_id = id if id_mapping == None else id_mapping[id]
        index = encoder.getIndex(real_id)
        name = getIndexName(index, mapping)
        print("Id:", id, "Name:", name)

In [11]:
def evaluateModel(name, config):
    encoder = IndicesClassEncoderEq(config["classes"], list(range(1, 12)))
    print("Total set length:", encoder.total_length)

    inform_cache = {}
    indep_cache = {}

    args = { 
        "num_generations":100, 
        "num_parents_mating":3,
        "parent_selection_type":"sss",
        "keep_elitism":1,
        "sol_per_pop":1000,
		"mutation_probability":0.1,
        "parallel_processing":8
    }

    eval_results = {}
    counter = 0
    for informativeness_threshold in config["thresholds"]["informativeness"]:
        for independency_threshold in config["thresholds"]["independency"]:
            opt = FeatureOptimizer(encoder, config["max_feature_count"],
                    funcs.bhattacharyya_distance, 
                    funcs.spearman_independency, 
                    optimization_method="genetic",
                    optimizer_args=args,
                    informativeness_threshold=informativeness_threshold, 
                    independency_threshold=independency_threshold,
                    set_independency=config["smoother"])
            
            opt.informativeness_cache = inform_cache
            opt.independency_cache = indep_cache


            if (name == "BANDS"):
                opt.fit(data, data[1], False, False)
                opt.selected_features = list(range(encoder.total_length))
            else:
                opt.fit(data, data[1], False)

            local_results = {}
            local_results["fitness"] = opt.get_fitness_()
            local_results["features"] = [int(f) for f in opt.selected_features]
            local_results["names"] = [opt.indicesEncoder.getIndex(f).args for f in opt.selected_features]
            

            for key in local_results.keys():
                print(key, local_results[key])

            print()
            display_features(opt.selected_features, encoder)

    return eval_results

In [12]:
tracemalloc.start()
time_start = time.time()

In [13]:
configurations = {}
configurations["NORMP"] = { "classes": [NORMP], "insert": False, "max_feature_count": 3, "thresholds": {"informativeness": [0.05], "independency": [0.05]}, "smoother": "geometric_mean", "models": 1 }

results = {}
for key in configurations.keys():
    results[key] = evaluateModel(key, configurations[key])

Total set length: 121


e:\Work\SWIFTT-BTI-2\.conda\lib\site-packages\pygad\pygad.py:1139: UserWarning: The 'delay_after_gen' parameter is deprecated starting from PyGAD 3.3.0. To delay or pause the evolution after each generation, assign a callback function/method to the 'on_generation' parameter to adds some time delay.
  warnings.warn("The 'delay_after_gen' parameter is deprecated starting from PyGAD 3.3.0. To delay or pause the evolution after each generation, assign a callback function/method to the 'on_generation' parameter to adds some time delay.")


Fitness (Gen 100): 1.0383269794174017
fitness 1.0383269794174017
features [13, 113, 97]
names [[3, 2], [4, 11], [10, 9]]

Id: 13 Name: NORMP(B4, B3)
Id: 113 Name: NORMP(B5, B12)
Id: 97 Name: NORMP(B11, B9)


In [14]:
time_end = time.time()
print("Time:", time_end - time_start, "sec")
print("MEM usage:", np.array(tracemalloc.get_traced_memory()) / 1024**2, "mb")
tracemalloc.stop()

Time: 15.182831525802612 sec
MEM usage: [ 0.59446526 16.16785908] mb
